# Week 3, day 4 (morning) — Worksheet 06 SOLUTIONS: business rules and standardization

Executed in the lab image. Every quoted number is what it actually printed.

Questions 6 and 7 are the ones to argue about in class. The cancelled-enrollment
rule moves two of three headline metrics by about 5% and leaves the third almost
untouched — which is not the result most people predict.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 06 — Business rules and standardization. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr = load("enrollment")
tx = load("transaction")
stu = load("students")
cat = load("category")
dtype = load("discount_type")

print("staged, uncleaned:")
print("  enrollment  ", enr.shape)
print("  transaction ", tx.shape)
print("  students    ", stu.shape)

PART A — slide 36's five rules

### Question 1

Rule 1 and 2: the paid-in-full flag. Print every distinct value of `transaction.full_paid` with its count, including nulls, and its dtype.
> **NOTE:** count the distinct representations before you write the mapping. Guessing there are two is how a rule ships with a hole in it.

In [ ]:
print("dtype:", tx.full_paid.dtype)
print()
print(tx.full_paid.value_counts(dropna=False).rename("rows").to_string())
print()
print("distinct representations:", tx.full_paid.nunique(dropna=False))
print("rows with a null flag:   ", int(tx.full_paid.isna().sum()))

```
dtype: str

full_paid
N      865
0      860
No     858
NaN    854
Y      510
1      485
Yes    424

distinct representations: 7
rows with a null flag:    854
```

**Seven representations of a two-state flag**, and the third most common value is
`NaN`.

Slide 36 lists six of them — `Y`, `Yes`, `1`, `N`, `No`, `0` — and stops there.
The seventh, null, is 854 rows: **17.6% of the table**, more than any single
non-null value except `N`. A rule written from the slide alone handles six cases
and leaves the largest one undecided.

This is why the question insists on counting before mapping. The instinct is to
write `full_paid == 'Y'` and move on, which is correct for 510 of 4,856 rows and
silently wrong for 909 more that say `Yes` or `1`. A `value_counts(dropna=False)`
on every flag column before writing any rule costs seconds and is the difference
between a rule and a guess.

Note `dtype: str`. Because one value is text, **the whole column is text** — so
`1` and `0` here are the *strings* `"1"` and `"0"`, not integers. Any comparison
like `full_paid == 1` returns `False` for all 4,856 rows, without error. Worksheet
04's Snowflake sibling had the same shape of trap with quoted values.

Where seven encodings come from is worth knowing: different application versions,
different data-entry paths, a migration that changed the convention, an import
that stringified booleans. Nobody chose this. It accumulated, and it will keep
accumulating — which is why question 2's mapping needs an explicit "unmapped"
branch rather than a default.

### Question 2

Standardize it. Map `Y`/`Yes`/`1` to 1 and `N`/`No`/`0` to 0, decide what a null becomes, and print the result — plus a check that nothing was left unmapped.

In [ ]:
TRUTHY = {"Y", "Yes", "1", "y", "yes", "TRUE", "True"}
FALSY = {"N", "No", "0", "n", "no", "FALSE", "False"}

def standardize_flag(v):
    if pd.isna(v):
        return 0                 # absent evidence of payment -> not paid
    s = str(v).strip()
    if s in TRUTHY:
        return 1
    if s in FALSY:
        return 0
    return None                  # unmapped -> must be zero of these

clean = tx.full_paid.map(standardize_flag)
print("standardized:")
print(clean.value_counts(dropna=False).rename("rows").to_string())
print()
print("unmapped values remaining:", int(clean.isna().sum()))
print("rows that were null and are now 0:", int(tx.full_paid.isna().sum()))

```
standardized:
full_paid
0    3437
1    1419

unmapped values remaining: 0
rows that were null and are now 0: 854
```

Seven values become two. 1,419 paid, 3,437 not.

Three design choices in that function, each worth defending.

**Nulls become 0, not null.** This is a business decision disguised as a
technical one. A null `full_paid` means *we have no record that this was paid in
full* — and for a payment flag, absence of evidence is evidence of absence. If
it were a *measurement* rather than a claim, the answer would be different: a
null temperature reading is not zero degrees.

The consequence is real. `is_paid_in_full` becomes a clean 0/1 column with no
third state, so `AVG(is_paid_in_full)` is a valid rate and `SUM` is a valid count.
Leave the nulls in and every downstream query has to decide again, and they will
not all decide the same way. **Resolve it once, at load, and write down that you
did.**

**The mapping is set-based and case-tolerant.** `TRUTHY` and `FALSY` include
`y`, `yes`, `TRUE` and their negatives — values not in this extract. That is
deliberate: the seven representations here are the seven that exist *today*, and
the eighth arrives with the next source release.

**Unmapped returns `None`, not 0.** This is the important one. A default of "0 if
not truthy" would silently absorb any new value — a `PENDING`, a `REFUNDED`, a
stray whitespace variant — into "not paid". Returning `None` makes an unknown
value **visible**, and `unmapped values remaining: 0` is then a real assertion
rather than a tautology. Worksheet 10 turns that line into a data-quality check.

The general rule: **standardization must fail loudly on the unexpected.** A rule
with a catch-all default cannot tell you it has stopped working.

### Question 3

Rule 3: the missing discount. Print `discount_type` in full, then count the enrollments that end up with a null `discount_amount` after joining. Break that count down by *why* it is null.
> **NOTE:** there is more than one way to arrive at a null discount here. Find both before writing the rule.

In [ ]:
print(dtype.to_string(index=False))
print()
per_enr = tx.groupby("enrl_id").discount_type_id.max().reset_index()
j = per_enr.merge(dtype[["discount_type_id", "discount_amount"]],
                  on="discount_type_id", how="left")

no_promo = j.discount_type_id.isna()
legacy = j.discount_type_id == 60
print("enrollments:                          %5d" % len(j))
print("null discount_amount after the join:  %5d" % int(j.discount_amount.isna().sum()))
print()
print("  because the enrollment had NO promotion:  %5d" % int(no_promo.sum()))
print("  because 'Legacy Promotion' has no amount: %5d" % int(legacy.sum()))
print("  the two together:                         %5d"
      % int((no_promo | legacy).sum()))
print()
j["discount_amount"] = j.discount_amount.fillna(0.0)
print("after the rule, nulls remaining:      %5d" % int(j.discount_amount.isna().sum()))
print("SUM(discount_amount):            %10.2f" % j.discount_amount.sum())

```
 discount_id  discount_type_id discount_type_name  discount_amount
           1                10         Early Bird            500.0
           2                20    Alumni Referral            750.0
           3                30   Employer Partner           1200.0
           4                40        Scholarship           2000.0
           5                50    Spring Campaign            300.0
           6                60   Legacy Promotion              NaN

enrollments:                           2243
null discount_amount after the join:   1216

  because the enrollment had NO promotion:   1019
  because 'Legacy Promotion' has no amount:   197
  the two together:                          1216

after the rule, nulls remaining:          0
SUM(discount_amount):             970700.00
```

**1,216 of 2,243 enrollments — 54% — end up with a null discount, from two
completely different causes.**

**1,019 had no promotion at all.** Their `discount_type_id` is null, so the left
join finds nothing. This is not missing data; it is a real business fact — most
students pay list price — and the correct discount is genuinely 0.

**197 used `Legacy Promotion`**, which exists in the lookup table but has no
amount recorded. This *is* missing data: the promotion was applied and nobody
knows what it was worth. Slide 36's rule says default it to 0.

They produce identical nulls and identical `fillna(0)` output, and they are not
the same thing. The first is 0 because zero is correct. The second is 0 because
**someone decided to treat unknown as zero**, which understates discounts by
whatever those 197 promotions were actually worth.

That second decision deserves to be visible. Two better options than a silent
default:

- **Flag it.** Add `discount_is_estimated` so the 197 rows are identifiable, and
  anyone computing a discount rate can see how much of it rests on an assumption.
- **Escalate it.** 197 enrollments with an unpriced promotion is a question for
  whoever maintains `discount_type`, and it is answerable — the value exists
  somewhere.

Note the trap the callout warned about. `SUM()` skips nulls in pandas and in SQL,
so totalling `discount_amount` before and after the rule gives the same answer.
**The rule changes nothing you can see with `SUM`** — and everything about
`net_tuition_amount = tuition - discount`, which is arithmetic, and arithmetic
does not skip nulls. Question 10 is that failure.

### Question 4

Rule 4: the missing category. Apply *missing category -> Unknown*, then count enrollments per category with and without the rule.

In [ ]:
crs, prg = load("course"), load("program")
chain = (enr.merge(crs[["course_id", "program_id"]], on="course_id")
            .merge(prg[["program_id", "category_id"]], on="program_id")
            .merge(cat[["category_id", "category_name"]], on="category_id",
                   how="left"))
raw = chain.groupby("category_name").size()
print("WITHOUT the rule (groupby drops nulls):")
print(raw.sort_values(ascending=False).rename("enrollments").to_string())
print("  total reported:", int(raw.sum()), "of", len(chain))
print()
chain["category_name"] = chain["category_name"].fillna("Unknown")
fixed = chain.groupby("category_name").size()
print("WITH the rule:")
print(fixed.sort_values(ascending=False).rename("enrollments").to_string())
print("  total reported:", int(fixed.sum()), "of", len(chain))

```
WITHOUT the rule (groupby drops nulls):
category_name
Cloud Computing     616
Data Science        603
Data Engineering    587
Cybersecurity       278
  total reported: 2084 of 2400

WITH the rule:
category_name
Cloud Computing     616
Data Science        603
Data Engineering    587
Unknown             316
Cybersecurity       278
  total reported: 2400 of 2400
```

One `fillna` and the report goes from 2,084 rows to 2,400.

This is worksheet 01 question 4's silent loss, closed. And notice what `Unknown`
turns out to be: **316 enrollments, 13% of the business, the fourth-largest
category** — bigger than Cybersecurity. That is not a rounding error hiding in a
null; it is a significant chunk of the business that no category-level report was
showing.

The `Unknown` label does three things a null cannot:

**It appears.** `groupby` keeps it, charts plot it, totals include it. The report
now adds up to the enrollment count, which means anyone checking can check.

**It is countable.** "13% of enrollments are uncategorised" is a statement
someone can act on. A null is not a value, so it cannot be measured — it can only
be absent.

**It provokes the right question.** Somebody sees `Unknown` at 316 and asks why,
and the answer is that `category_id = 5` has no name and the `Foundations
Bootcamp` program points at it. That is a ten-minute fix in the source system,
and it never gets made while the rows are invisible.

This is the general pattern for missing dimension attributes, and it extends to
keys: give every dimension an `Unknown` member so unmatched facts land somewhere
countable rather than being dropped by a join. Worksheet 09 does that with
`student_id = -1`.

**Never let a null silently remove rows from a report.** Replace it with a label
that says what it is.

### Question 5

Rule 5, part one: exclude the non-business rows. Remove enrollments belonging to TEST students and those referencing an unknown `stu_id`, and print the count removed and remaining.

In [ ]:
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
known = set(stu.stu_id)

is_test = enr.stu_id.isin(test_ids)
is_orphan = ~enr.stu_id.isin(known)
print("TEST-student enrollments:   ", int(is_test.sum()))
print("unknown-student enrollments:", int(is_orphan.sum()))
print("overlap:                    ", int((is_test & is_orphan).sum()))
print()
kept = enr[~(is_test | is_orphan)]
print("before: %d   removed: %d   after: %d"
      % (len(enr), len(enr) - len(kept), len(kept)))

```
TEST-student enrollments:    18
unknown-student enrollments: 7
overlap:                     0

before: 2400   removed: 25   after: 2375
```

Twenty-five rows removed, and the two rules do not overlap — these are distinct
problems that happen to share a fix.

**The 18 TEST enrollments** are QA artefacts and should not be in a business
report. But look at how they were found: `stu_name.str.startswith("TEST")`. That
is string-matching on a free-text field, and it is fragile in both directions —
it misses a test account named `Jane Smith`, and it would wrongly exclude a real
student named `TESTA Ndlovu`. It works here because the generator was
consistent; it is not a technique to be proud of.

The real fix is upstream: a flag on the account, or a separate test environment
that never reaches the extract. Where that is not available, the string rule is
the right pragmatic answer **provided it is written down and its count is
monitored**. If `TEST` enrollments jump from 18 to 400 next month, something
changed and you want to know.

**The 7 unknown-student enrollments** are a different animal — a referential
integrity failure. The enrollment references a `stu_id` that is not in
`students`, which means either the two tables were extracted at different moments
or the source deleted a student who had enrollments.

Excluding them is the *convenient* answer and probably the wrong one. It makes a
real inconsistency disappear from view: next month it might be 700 rows, and the
pipeline will quietly drop those too. Worksheet 09 routes them to an `Unknown`
student instead, so they stay counted and stay visible, and worksheet 10 makes
their count a monitored check.

The distinction is worth stating generally. **Exclude rows that are not business
events. Flag rows that are business events with a problem.** A test account is
the first. A real enrollment with a broken foreign key is the second.

PART B — the rule with two answers

### Question 6

Rule 5, part two. Slide 36 says cancelled enrollments are *"excluded or flagged"*. Build both: an excluded version and a flagged version. Print the row count of each and the enrollment count they report.

In [ ]:
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
base = enr[~enr.stu_id.isin(test_ids) & enr.stu_id.isin(set(stu.stu_id))].copy()

excluded = base[base.status != "cancelled"]
flagged = base.copy()
flagged["is_cancelled"] = (flagged.status == "cancelled").astype(int)

print("after the question-5 filters:      %5d rows" % len(base))
print()
print("OPTION A -- excluded:              %5d rows" % len(excluded))
print("  enrollments reported:            %5d" % len(excluded))
print()
print("OPTION B -- flagged:               %5d rows" % len(flagged))
print("  enrollments reported (all):      %5d" % len(flagged))
print("  enrollments reported (active):   %5d"
      % int((flagged.is_cancelled == 0).sum()))
print("  cancellation rate:               %5.2f%%"
      % (100 * flagged.is_cancelled.mean()))

```
after the question-5 filters:       2375 rows

OPTION A -- excluded:               2259 rows
  enrollments reported:             2259

OPTION B -- flagged:                2375 rows
  enrollments reported (all):       2375
  enrollments reported (active):    2259
  cancellation rate:                 4.88%
```

Both options are correct implementations of slide 36. They differ by **116 rows**
and by something more important: what questions remain askable.

**Option A deletes information.** The 2,259 rows are the active enrollments, and
that is the right number for a revenue report. But the cancellations are gone —
not filtered, *gone* — so "what is our cancellation rate?" cannot be answered from
this fact table at all. Answering it means going back to the source.

**Option B keeps everything and adds a column.** 2,375 rows, a 0/1 flag, and now
both questions work:

```sql
-- active enrollments
SELECT SUM(enrollment_count) FROM fact_enrollment WHERE is_cancelled = 0;
-- cancellation rate
SELECT AVG(is_cancelled) FROM fact_enrollment;
```

**4.88% is a metric the business almost certainly wants**, and Option A makes it
unavailable while looking like a tidier table.

The general principle, and it is one of the most useful in dimensional modelling:

> **Flag, do not filter.** A filter applied at load time is a decision imposed on
> every future question. A flag is a decision offered to each question
> separately.

The cost of flagging is that every query must remember the `WHERE is_cancelled =
0`, and some will forget. That is a real cost and it has a real answer: publish a
**view** — `v_active_enrollments` — that applies the filter, and point casual
users at the view while leaving the full fact table available for the questions
that need it. You get the safe default and you keep the data.

Filter only when the rows are genuinely not business events — the TEST accounts
in question 5. A cancelled enrollment is a thing that happened.

### Question 7

Show what the choice actually costs. Compute three headline metrics under both options: the enrollment count, the total amount paid, and the full-payment rate. Print the percentage change in each.
> **NOTE:** predict which of the three moves most before you run it. The answer is not the same for all three, and the reason why is the point.

In [ ]:
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
base = enr[~enr.stu_id.isin(test_ids) & enr.stu_id.isin(set(stu.stu_id))].copy()

TRUTHY = {"Y", "Yes", "1"}
paid = (tx.assign(flag=tx.full_paid.map(
            lambda v: 0 if pd.isna(v) else int(str(v).strip() in TRUTHY)))
          .groupby("enrl_id").flag.max())
base = base.merge(paid.rename("is_paid_in_full"), on="enrl_id", how="left")
base["is_paid_in_full"] = base["is_paid_in_full"].fillna(0).astype(int)

paid_amt = tx.groupby("enrl_id").payment_amount.sum()
base = base.merge(paid_amt.rename("amount_paid"), on="enrl_id", how="left")
base["amount_paid"] = base["amount_paid"].fillna(0.0)
excl = base[base.status != "cancelled"]

metrics = [
    ("enrollment count", len(excl), len(base)),
    ("total amount paid", excl.amount_paid.sum(), base.amount_paid.sum()),
    ("full-payment rate", 100 * excl.is_paid_in_full.mean(),
     100 * base.is_paid_in_full.mean()),
]
print("%-20s %14s %14s %10s" % ("METRIC", "A: excluded", "B: included", "CHANGE"))
for name, a, b in metrics:
    print("%-20s %14.2f %14.2f %9.2f%%" % (name, a, b, 100 * (a - b) / b))

```
METRIC                  A: excluded    B: included     CHANGE
enrollment count            2259.00        2375.00     -4.88%
total amount paid        8407463.58     8859405.79     -5.10%
full-payment rate             58.70          58.86     -0.28%
```

Two metrics move by about 5%. The third barely moves at all — **0.28%** — and
that asymmetry is the lesson.

The reason is what the rule touches:

**Enrollment count** is a pure count of rows. Remove 116 rows and it falls by
exactly those 116 — 4.88%. The rule changes the numerator directly.

**Total amount paid** is a sum over rows. It falls by 5.10%, slightly more than
the row count, because the 116 cancelled enrollments had paid slightly more than
average before cancelling. Again, the rule changes the numerator.

**Full-payment rate** is a ratio, and the rule shrinks **both** the numerator and
the denominator by nearly the same proportion. 1,326/2,259 against 1,398/2,375 —
the two effects almost cancel, and 58.70% versus 58.86% is a difference nobody
would notice.

So "does this rule matter?" has no single answer. It depends entirely on which
metric you are looking at, and a rule that is invisible in one report can move
another by 5%.

Two practical consequences.

**Test a rule against the metrics it will actually affect, not just one.** Had
this worksheet checked only the full-payment rate, the honest conclusion would
have been "the cancelled rule barely matters" — and revenue would have been off
by 452,000.

**Ratios are the most deceptive place to look for a rule's impact**, because
correlated changes to numerator and denominator hide it. When validating a
pipeline change, compare counts and sums first; check ratios last, and never
treat a stable ratio as evidence that nothing changed.

Note what this does *not* show: that cancelled enrollments are unusual. Their
full-payment rate is close to the overall rate, which is mildly surprising —
people who cancel here had often already paid. Whether that is realistic is a
question for the business, and it is the sort of thing this comparison surfaces
for free.

PART C — order, and writing it down

### Question 8

Rule order matters. Apply *drop cancelled* then *count distinct students*, and then the same two in the other order. Print both results.
> **NOTE:** filtering and aggregating do not commute. One of these answers a different question from the one asked.

In [ ]:
base = enr.copy()
a = base[base.status != "cancelled"].stu_id.nunique()
print("filter THEN count distinct students: %d" % a)

per_student = base.groupby("stu_id").status.apply(
    lambda s: (s != "cancelled").any())
b = int(per_student.sum())
print("count students with ANY active enrollment: %d" % b)
print()
print("students appearing only in cancelled enrollments: %d" % (
    base.stu_id.nunique() - b))
print("total distinct students in the raw table:        %d" % base.stu_id.nunique())

```
filter THEN count distinct students: 599
count students with ANY active enrollment: 599

students appearing only in cancelled enrollments: 3
total distinct students in the raw table:        602
```

Both routes give 599, and the reason they agree is worth more than the number.

`filter THEN count distinct` asks: *how many distinct students appear among the
active enrollments?* `count students with ANY active enrollment` asks: *how many
students have at least one active enrollment?* On this data those are the same
599, because a student who has any active enrollment appears in the filtered set
by definition.

Where they would diverge is the reverse pairing — **aggregate first, then
filter**:

```python
# students whose enrollments are ALL cancelled
base.groupby("stu_id").status.apply(lambda s: (s == "cancelled").all()).sum()
```

which is **3**, and no amount of filtering-then-counting produces it. The 3 are
the students who tried and cancelled everything, and they are invisible in the
599 either way.

The general rule: **filtering and aggregating do not commute, and the difference
is which population you are describing.** 602 students exist. 599 have an active
enrollment. 3 have only cancellations. Each is a valid answer to a differently
worded question, and the wording differences are small enough to be lost in
translation between an analyst and a stakeholder.

This is why slide 41 asks for *"target grain and aggregation logic"* in the
specification, and why "distinct students" is never a sufficient metric
definition. **Distinct students in what set, at what grain, after which filters?**
Every one of those clauses changes the number.

The practical habit: when a metric involves `DISTINCT` and a filter, write out
the population in words before writing the SQL, and put that sentence next to the
metric where the next person will read it.

### Question 9

Write the rules down. Build a rule register — id, target column, condition, action, rows affected — and print it, computing `rows affected` from the data rather than typing it.
> **NOTE:** slide 41 asks an ETL spec to document *"business rules and standardization logic"*. A register that computes its own counts cannot drift from the code.

In [ ]:
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
known = set(stu.stu_id)
crs, prg = load("course"), load("program")
chain = (enr.merge(crs[["course_id", "program_id"]], on="course_id")
            .merge(prg[["program_id", "category_id"]], on="program_id")
            .merge(cat[["category_id", "category_name"]], on="category_id",
                   how="left"))

RULES = [
    ("R1", "is_paid_in_full", "full_paid in Y/Yes/1", "-> 1",
     int(tx.full_paid.isin(["Y", "Yes", "1"]).sum())),
    ("R2", "is_paid_in_full", "full_paid null", "-> 0",
     int(tx.full_paid.isna().sum())),
    ("R3", "discount_amount", "no promotion, or null amount", "-> 0.00",
     int(tx.groupby("enrl_id").discount_type_id.max().reset_index()
           .merge(dtype[["discount_type_id", "discount_amount"]],
                  on="discount_type_id", how="left")
           .discount_amount.isna().sum())),
    ("R4", "program_category", "category_name null", "-> 'Unknown'",
     int(chain.category_name.isna().sum())),
    ("R5", "(row)", "student name starts TEST", "exclude",
     int(enr.stu_id.isin(test_ids).sum())),
    ("R6", "(row)", "stu_id not in students", "exclude",
     int((~enr.stu_id.isin(known)).sum())),
    ("R7", "is_cancelled", "status = 'cancelled'", "flag, do not exclude",
     int((enr.status == "cancelled").sum())),
]
print("%-4s %-17s %-30s %-22s %s" % ("ID", "TARGET", "CONDITION", "ACTION", "ROWS"))
for r in RULES:
    print("%-4s %-17s %-30s %-22s %5d" % r)

```
ID   TARGET            CONDITION                      ACTION                 ROWS
R1   is_paid_in_full   full_paid in Y/Yes/1           -> 1                    1419
R2   is_paid_in_full   full_paid null                 -> 0                     854
R3   discount_amount   no promotion, or null amount   -> 0.00                 1216
R4   program_category  category_name null             -> 'Unknown'             316
R5   (row)             student name starts TEST       exclude                   18
R6   (row)             stu_id not in students         exclude                    7
R7   is_cancelled      status = 'cancelled'           flag, do not exclude     117
```

Seven rules, and every row count is **computed from the data** rather than typed
in. That is the whole point of the exercise.

A rule register written in a wiki page is accurate on the day it is written. This
one cannot drift: if the source stops producing `Yes`, R1's count changes on the
next run; if a new promotion arrives without an amount, R3's count grows. The
document and the data cannot disagree, because the document reads the data.

Slide 41 asks an ETL specification to record *"business rules and standardization
logic"*, and the practical form that should take is a table like this one,
regenerated with the pipeline and stored beside the output. Three things it gives
you:

**A regression signal.** These seven numbers are a fingerprint of the source. If
R2 jumps from 854 to 3,000 overnight, an upstream system changed how it writes
that flag — and you find out from the register rather than from a stakeholder.

**A review artefact.** R7's *"flag, do not exclude"* is a business decision, and
it is now written where a business person can read it and object. That
conversation is much cheaper before the dashboard exists.

**An audit trail.** When a number is challenged, the question is always "what did
you do to the data?" This answers it in seven lines.

Note that R1 and R2 are separate rules, not one. They map to the same target
column and they are different decisions — R1 restates the source's own claim, R2
invents a value the source did not provide. **Rules that assume should be
separately visible from rules that translate**, because they carry different
risk and only one of them is worth escalating.

### Question 10

Finally, skip rule R3 and compute `net_tuition_amount = tuition - discount` with the null still in place, then assert the result has no nulls. **This is supposed to fail.** Read the error and say why arithmetic did not raise on its own.

In [ ]:
per_enr = (tx.groupby("enrl_id")
             .agg(tuition=("full_price", "max"),
                  discount_type_id=("discount_type_id", "max"))
             .reset_index())
joined = per_enr.merge(dtype[["discount_type_id", "discount_amount"]],
                       on="discount_type_id", how="left")
joined["net_tuition"] = joined.tuition - joined.discount_amount

print("enrollments:                 ", len(joined))
print("null discount_amount:        ", int(joined.discount_amount.isna().sum()))
print("null net_tuition:            ", int(joined.net_tuition.isna().sum()))
print("SUM(net_tuition):  %14.2f  <- computed without complaint"
      % joined.net_tuition.sum())
print()
assert joined.net_tuition.notna().all(), \
    "net_tuition has %d nulls -- rule R3 was not applied" % int(
        joined.net_tuition.isna().sum())

```
enrollments:                  2243
null discount_amount:         1216
null net_tuition:             1216
SUM(net_tuition):      4357200.00  <- computed without complaint

AssertionError: net_tuition has 1216 nulls -- rule R3 was not applied
```

Look at the order of those lines. The **arithmetic ran fine.** `tuition -
discount_amount` produced 1,216 nulls without a word, and `SUM` then reported
4,357,200.00 by skipping every one of them. Only the explicit assertion objected.

This is null propagation, and it is consistent across pandas and SQL: any
arithmetic involving null yields null, and aggregates skip nulls. Both behaviours
are individually sensible and together they are a silent-corruption engine.

**54% of the fact table has no net tuition, and the total looks plausible.** It is
plausible because it is a real sum of real numbers — just of 1,027 rows instead of
2,243. A revenue figure computed over less than half the business, with no error,
no warning, and no obvious sign in the output.

Worse, the failure is *stable*. It will produce a similar-looking number every
night, trending sensibly, until someone reconciles against the source.

Two defences, and you need both.

**Apply the rule** — `fillna(0)` before the arithmetic, which is R3 and costs one
line.

**Assert that it was applied.** The rule and the check are separate artefacts.
`assert net_tuition.notna().all()` does not trust that the earlier line ran, that
it ran on this column, or that a refactor left it in place. Worksheet 10 makes
this one of slide 39's standard checks — *"are required keys or measures
missing?"*

The habit worth forming: **after every derived column, assert its null count is
what you expect** — usually zero, occasionally a known number. It is one line per
column, it runs in milliseconds, and it converts the most common silent failure in
data engineering into a loud one.

**What this sheet established:**

| | |
|---|---|
| slide 36's two-state flag | **7 representations**, including 854 nulls the slide does not mention |
| standardized | 7 values -> 2, with unmapped values surfaced rather than defaulted |
| null discounts | **1,216** enrollments, from **two different causes** that need the same fix and different scrutiny |
| missing category | `Unknown` is **316 enrollments — 13%**, the fourth-largest category |
| non-business rows | 18 TEST + 7 orphans, no overlap; exclude the first, flag the second |
| cancelled: exclude or flag | 2,259 vs 2,375 rows — flagging keeps a **4.88% cancellation rate** answerable |
| the rule's real impact | count **-4.88%**, revenue **-5.10%**, but the rate only **-0.28%** |
| skipping a rule | 1,216 nulls, no error, and a revenue total over 46% of the business |

Worksheet 07 is Step 4: turning these cleaned columns into the five measures and
the flag that `fact_enrollment` actually carries.